# Pilot B live check: graph-edit skill, Gemma 4 E4B, with vs without

Attach: competition `gemma-4-developer-agent`, dataset `metric/gemma-4-developer-agent-wheelhouse`,
model `google/gemma-4/Other/gemma-4-e4b-it-qat-w4a16-ct/2`. Accelerator: **GPU L4 x4**. Internet off.

1. Run with `SMOKE = True` (3 tasks x 2 conditions). It prints measured minutes and an extrapolation.
2. If the extrapolation is OK, set `SMOKE = False` and run again (15 tasks x 2 conditions).
3. Download only `/kaggle/working/pilot_b_live_results.json` (aggregate numbers + per-task ids/numbers;
   no code, patches or traces) and commit it to `results/` on the pilot branch.

In [ ]:
SMOKE = True
MODEL_NAME = 'gemma-4-e4b-it-qat-w4a16-ct'
TASK_IDS = ['fastapi_14458', 'rich_3518', 'requests_7502'] if SMOKE else ['fastapi_14458', 'fastapi_14964', 'rich_3518', 'rich_3535', 'fastapi_13920', 'fastapi_14266', 'requests_6644', 'rich_3130', 'fastapi_14303', 'fastapi_15745', 'requests_7502', 'rich_3882', 'fastapi_5077', 'rich_3506', 'rich_3777']
CONDITIONS = ['baseline', 'graph_edit']
# identical budgets for both conditions (the sample's 10 calls / 1 minute is a toy budget)
BUDGET = dict(max_tool_calls=50, max_time_minutes=10.0, timeout_seconds=120, max_turns=150)
HARD_STOP_H = 1.9   # never start a condition after this much notebook wall time (cap: 2 L4x4 h)
CONCURRENCY = 4
SEED = 1234
import time; T0 = time.time()

In [ ]:
import glob, importlib, logging, os, shutil, subprocess, sys, re, json
from pathlib import Path
for k, v in {'TRANSFORMERS_NO_TF': '1', 'TRANSFORMERS_NO_FLAX': '1', 'TRANSFORMERS_OFFLINE': '1',
             'VLLM_NO_USAGE_STATS': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn', 'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
             'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1', 'NCCL_DEBUG': 'WARN', 'LITELLM_LOCAL_MODEL_COST_MAP': 'True',
             'LITELLM_TELEMETRY': 'False', 'LITELLM_LOG': 'WARNING', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
             'OTEL_SDK_DISABLED': 'true', 'OTEL_PYTHON_DISABLED': 'true'}.items():
    os.environ[k] = v
os.environ.setdefault('OPENAI_API_KEY', 'EMPTY'); os.environ.setdefault('LOCAL_API_KEY', 'EMPTY')
for name in ('LiteLLM', 'LiteLLM Router', 'LiteLLM Proxy', 'litellm', 'httpx', 'httpcore', 'openai', 'urllib3', 'opentelemetry'):
    lg = logging.getLogger(name); lg.setLevel(logging.WARNING)
    if 'litellm' in name.lower(): lg.propagate = False; lg.handlers.clear()
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    t = tmp_whl / (w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name)
    if not t.exists():
        try: os.symlink(w, t)
        except OSError: shutil.copyfile(w, t)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *sorted(map(str, tmp_whl.glob('*.whl')))], check=True)
importlib.invalidate_caches()
print('wheelhouse installed')

In [ ]:
# Embedded pilot files (generated from the repo by pilot_b/build_notebook.py)
PILOT = Path('/kaggle/working/pilot_b'); PILOT.mkdir(parents=True, exist_ok=True)
EMBEDDED = {
"skills/graph-edit/SKILL.md": "---\nname: graph-edit\ndescription: View and change Python code by symbol name (pkg.module.Class.method) instead of copying exact text. Use it to read a function/class and to replace or add whole functions, methods or classes. The edit is syntax-checked before it is written.\n---\n# graph-edit: edit code by symbol name\n\nSymbol names are the dotted names the code-graph tools print, e.g.\n`fastapi.routing.APIRoute.__init__`. You can also write `path/to/file.py:Class.method`\nor a unique short name such as `APIRoute.__init__`.\n\n## 1. Look at a symbol\nrun_skill_script(skill_name=\"graph-edit\", file_path=\"scripts/show_symbol.py\",\n                 args={\"symbol\": \"fastapi.routing.APIRoute.__init__\"})\nReturns JSON: file, line range, and the full `source`. Pass a file path\n(`args={\"symbol\": \"fastapi/routing.py\"}`) to list every symbol in that file.\n\n## 2. Replace a whole function, method or class\nrun_skill_script(skill_name=\"graph-edit\", file_path=\"scripts/edit_symbol.py\",\n                 args={\"symbol\": \"fastapi.routing.APIRoute.__init__\",\n                       \"new_source\": \"def __init__(self, path: str) -> None:\\n    self.path = path\\n\"})\n- `new_source` is the COMPLETE new definition (def line + full body), not a fragment.\n- Indentation is fixed for you: write it starting at column 0.\n- Decorators: if you leave them out, the existing decorators are kept.\n- The file is only written if the result is valid Python. On error you get\n  `\"status\": \"error\"` and the file is unchanged; fix `new_source` and retry.\n- The reply contains a short diff of what changed. Check it.\n\n## 3. Add a NEW function, method or class\nrun_skill_script(skill_name=\"graph-edit\", file_path=\"scripts/insert_symbol.py\",\n                 args={\"new_source\": \"def helper(x):\\n    return x\\n\",\n                       \"after\": \"fastapi.routing.APIRoute.__init__\"})\nUse exactly one of `after`, `before`, or `into` (a class name, or a file path to append at the end).\n\n## When NOT to use it\nImports and other module-level lines (constants, `__all__`) are not symbols: change those\nwith edit_file. Never edit test files.\n",
"skills/graph-edit/scripts/_symlib.py": "\"\"\"Symbol-addressed view/edit of Python source. Stdlib only; runs inside the task sandbox.\n\nSymbols are addressed by dotted name, as the code-graph tools print them\n(pkg.module.Class.method), or by `path/to/file.py:Class.method`, or by a unique suffix\n(`Class.method`). Both the true nested name and the graph's own (flattened) name resolve.\nSpans are recomputed from the current file with `ast` on every call, so earlier edits never\nleave stale offsets.\n\"\"\"\nfrom __future__ import annotations\n\nimport ast\nimport difflib\nimport json\nimport os\nimport re\nimport sys\nimport textwrap\n\nSKIP_DIRS = {\".git\", \"__pycache__\", \".venv\", \"venv\", \"node_modules\", \"build\", \"dist\", \".tox\",\n             \"site-packages\", \".pytest_cache\", \".mypy_cache\"}\nSTRIP_PREFIXES = (\"src/\", \"docs_src/\")\nMAX_OUT = 6000\nDEF_TYPES = (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)\nKIND = {ast.FunctionDef: \"def\", ast.AsyncFunctionDef: \"async def\", ast.ClassDef: \"class\"}\n\n\nclass SymError(Exception):\n    def __init__(self, error_type: str, message: str, **details):\n        super().__init__(message)\n        self.error_type, self.message, self.details = error_type, message, details\n\n\n# ---------------------------------------------------------------- workspace / files\ndef find_root() -> str:\n    cands = [os.environ.get(\"GRAPH_EDIT_ROOT\"), \"/workspace\"]\n    cands += (os.environ.get(\"PYTHONPATH\") or \"\").split(os.pathsep)\n    cands += [os.environ.get(\"PWD\"), os.getcwd()]\n    for c in cands:\n        if c and os.path.exists(os.path.join(c, \".git\")):\n            return os.path.abspath(c)\n    raise SymError(\"WorkspaceNotFound\", \"could not locate the repository root (/workspace)\")\n\n\ndef module_name(rel: str) -> str:\n    p = rel\n    for pre in STRIP_PREFIXES:\n        if p.startswith(pre):\n            p = p[len(pre):]\n            break\n    parts = (p[:-3] if p.endswith(\".py\") else p).split(\"/\")\n    if parts[-1] == \"__init__\":\n        parts = parts[:-1]\n    return \".\".join(parts)\n\n\ndef iter_py(root: str):\n    for dp, dns, fns in os.walk(root):\n        dns[:] = sorted(d for d in dns if d not in SKIP_DIRS and not d.startswith(\".\"))\n        for fn in sorted(fns):\n            if fn.endswith(\".py\"):\n                yield os.path.relpath(os.path.join(dp, fn), root).replace(os.sep, \"/\")\n\n\ndef read(root: str, rel: str) -> str:\n    with open(os.path.join(root, rel), encoding=\"utf-8\") as f:\n        return f.read()\n\n\ndef is_test_path(rel: str) -> bool:\n    b = os.path.basename(rel)\n    return b.startswith(\"test_\") or b.endswith(\"_test.py\") or b == \"conftest.py\" or \\\n        any(part in (\"tests\", \"test\", \"testing\") for part in rel.split(\"/\")[:-1])\n\n\ndef is_aux_path(rel: str) -> bool:\n    return rel.split(\"/\")[0] in (\"docs_src\", \"docs\", \"examples\", \"benchmarks\", \"scripts\", \"tests\", \"test\")\n\n\n# ---------------------------------------------------------------- symbol index\nclass Sym:\n    __slots__ = (\"qname\", \"gname\", \"rel\", \"node\", \"parent\", \"start\", \"end\", \"col\")\n\n    def __init__(self, qname, gname, rel, node, parent):\n        self.qname, self.gname, self.rel, self.node, self.parent = qname, gname, rel, node, parent\n        self.start = min([d.lineno for d in node.decorator_list] + [node.lineno])\n        self.end, self.col = node.end_lineno, node.col_offset\n\n    @property\n    def kind(self):\n        return KIND[type(self.node)]\n\n    def brief(self):\n        return {\"symbol\": self.qname, \"kind\": self.kind, \"file\": self.rel, \"lines\": [self.start, self.end]}\n\n\ndef index_file(root: str, rel: str, src: str | None = None) -> list[Sym]:\n    src = read(root, rel) if src is None else src\n    tree = ast.parse(src)\n    mod = module_name(rel)\n    out: list[Sym] = []\n    state = {\"cls\": None}  # emulates the competition graph's (flattening) naming\n\n    def visit(node, prefix, parent):\n        for ch in ast.iter_child_nodes(node):\n            if isinstance(ch, DEF_TYPES):\n                q = f\"{prefix}.{ch.name}\"\n                if isinstance(ch, ast.ClassDef):\n                    g = f\"{mod}.{ch.name}\"\n                else:\n                    g = f\"{mod}.{state['cls']}.{ch.name}\" if state[\"cls\"] else f\"{mod}.{ch.name}\"\n                s = Sym(q, g, rel, ch, parent)\n                out.append(s)\n                if isinstance(ch, ast.ClassDef):\n                    state[\"cls\"] = ch.name\n                    visit(ch, q, s)\n                    state[\"cls\"] = None\n                else:\n                    visit(ch, q, s)\n            else:\n                visit(ch, prefix, parent)\n\n    visit(tree, mod, None)\n    return out\n\n\ndef _split_symbol(symbol: str):\n    symbol = symbol.strip().strip(\"`'\\\"\")\n    m = re.match(r\"^(?:/?workspace/)?(?P<path>[\\w./-]+\\.py)\\s*(?:::?|#)\\s*(?P<rest>[\\w.]*)$\", symbol)\n    if m:\n        return m.group(\"path\").lstrip(\"/\"), m.group(\"rest\")\n    if symbol.endswith(\".py\"):\n        return symbol.lstrip(\"/\"), \"\"\n    return None, symbol\n\n\ndef resolve(root: str, symbol: str, kinds=None) -> Sym:\n    symbol = symbol.strip()\n    at_line = None\n    m = re.match(r\"^(.*?)@(\\d+)$\", symbol)            # name@<line> picks among same-named defs\n    if m:\n        symbol, at_line = m.group(1), int(m.group(2))\n    path, dotted = _split_symbol(symbol)\n    if path is not None:\n        if not os.path.isfile(os.path.join(root, path)):\n            raise SymError(\"FileNotFound\", f\"no such file: {path}\")\n        files = [path]\n        if not dotted:\n            raise SymError(\"NotASymbol\", \"a file was given; use show_symbol on the file to list its symbols\")\n        mod = module_name(path)\n        full = f\"{mod}.{dotted}\" if mod and not dotted.startswith(mod + \".\") else dotted\n        cands = [s for s in index_file(root, path)\n                 if full in (s.qname, s.gname) or s.qname.endswith(\".\" + dotted) or s.gname.endswith(\".\" + dotted)]\n    else:\n        full = dotted\n        parts = dotted.split(\".\")\n        files, cands = [], []\n        mods = {}\n        for rel in iter_py(root):\n            mods.setdefault(module_name(rel), []).append(rel)\n        for k in range(len(parts) - 1, 0, -1):          # longest module prefix wins\n            mod = \".\".join(parts[:k])\n            if mod in mods:\n                files = mods[mod]\n                for rel in files:\n                    cands += [s for s in _safe_index(root, rel) if dotted in (s.qname, s.gname)]\n                break\n        if not cands:                                     # suffix search across the repo\n            suffix = \".\" + dotted\n            for rel in sorted(set(r for v in mods.values() for r in v)):\n                for s in _safe_index(root, rel):\n                    if s.qname.endswith(suffix) or s.gname.endswith(suffix):\n                        cands.append(s)\n    if kinds:\n        cands = [c for c in cands if c.kind in kinds]\n    cands = list({(c.rel, c.start): c for c in cands}.values())\n    if at_line is not None:\n        hit = [c for c in cands if c.start <= at_line <= c.end]\n        hit.sort(key=lambda c: c.end - c.start)          # innermost containing the line\n        cands = hit[:1] or cands\n    # ranking: exact true name > exact graph name > suffix; then library > tests > docs/examples\n    for keep in (lambda c: c.qname == full, lambda c: c.gname == full,\n                 lambda c: not is_test_path(c.rel), lambda c: not is_aux_path(c.rel)):\n        if len(cands) > 1:\n            kept = [c for c in cands if keep(c)]\n            if kept:\n                cands = kept\n    if not cands:\n        near = _near_misses(root, dotted, files)\n        raise SymError(\"SymbolNotFound\", f\"no symbol named {symbol!r}\", did_you_mean=near)\n    if len(cands) > 1:\n        raise SymError(\"AmbiguousSymbol\", f\"{len(cands)} definitions match {symbol!r}; pass a full dotted name, \"\n                       \"file.py:Name, or append @<line> (a line inside the one you mean)\",\n                       candidates=[c.brief() for c in cands[:10]])\n    return cands[0]\n\n\ndef _safe_index(root, rel):\n    try:\n        return index_file(root, rel)\n    except (SyntaxError, UnicodeDecodeError, ValueError, OSError):\n        return []\n\n\ndef _near_misses(root, dotted, files):\n    last = dotted.split(\".\")[-1]\n    pool = []\n    for rel in (files or []):\n        pool += [s.qname for s in _safe_index(root, rel)]\n    if not pool:\n        for rel in iter_py(root):\n            if is_test_path(rel):\n                continue\n            pool += [s.qname for s in _safe_index(root, rel) if s.node.name.lower() == last.lower()\n                     or last.lower() in s.node.name.lower()]\n            if len(pool) > 4000:\n                break\n    close = difflib.get_close_matches(dotted, pool, n=5, cutoff=0.5)\n    return close or pool[:5]\n\n\n# ---------------------------------------------------------------- snippet normalisation\nFENCE = re.compile(r\"^\\s*```[\\w-]*\\s*$\")\n\n\ndef clean_snippet(code: str) -> str:\n    lines = code.replace(\"\\r\\n\", \"\\n\").split(\"\\n\")\n    while lines and not lines[0].strip():\n        lines.pop(0)\n    while lines and not lines[-1].strip():\n        lines.pop()\n    # strip a markdown fence only when it wraps the whole snippet (never inside docstrings)\n    if len(lines) >= 2 and FENCE.match(lines[0]) and FENCE.match(lines[-1]):\n        lines = lines[1:-1]\n    if not lines:\n        raise SymError(\"EmptySource\", \"new_source is empty\")\n    return \"\\n\".join(lines)\n\n\ndef _indent(line: str) -> int:\n    return len(line) - len(line.lstrip(\" \"))\n\n\ndef _norm_dump(tree) -> str:\n    class N(ast.NodeTransformer):\n        def visit_Constant(self, n):\n            if isinstance(n.value, str):\n                return ast.copy_location(ast.Constant(\" \".join(n.value.split())), n)\n            return n\n    return ast.dump(N().visit(tree))\n\n\ndef parse_snippet(code: str) -> tuple[list[str], ast.AST]:\n    \"\"\"Return (lines at column 0 with 4-space body indent where possible, the single def/class node).\"\"\"\n    code = clean_snippet(code)\n    lines = code.split(\"\\n\")\n    # \"graph text\" style: first line at col 0, body at its original deeper indentation\n    if len(lines) > 1 and _indent(lines[0]) == 0:\n        body_ind = [_indent(ln) for ln in lines[1:] if ln.strip()]\n        if body_ind and min(body_ind) > 0:\n            pass  # already parseable as-is; relative indentation fixed below\n    text = textwrap.dedent(\"\\n\".join(lines))\n    try:\n        tree = ast.parse(text)\n    except SyntaxError as e:\n        raise SymError(\"SyntaxError\", f\"new_source does not parse: {e.msg} (snippet line {e.lineno})\",\n                       line=e.lineno, text=(e.text or \"\").rstrip()[:200])\n    defs = [n for n in tree.body if isinstance(n, DEF_TYPES)]\n    if len(tree.body) != 1 or len(defs) != 1:\n        raise SymError(\"NotASingleDefinition\",\n                       \"new_source must contain exactly one def/async def/class (decorators allowed); \"\n                       f\"got {len(tree.body)} top-level statements\",\n                       top_level=[type(n).__name__ for n in tree.body][:10])\n    node = defs[0]\n    lines = text.split(\"\\n\")\n    # normalise an over-indented body (e.g. copied from graph/search output) to +4\n    if node.body:\n        b0 = node.body[0].lineno - 1\n        d = _indent(lines[b0])\n        region = [ln for ln in lines[b0:] if ln.strip()]\n        if d > 4 and all(_indent(ln) >= d for ln in region):\n            cand = lines[:b0] + [ln[d - 4:] if ln.strip() else ln for ln in lines[b0:]]\n            try:\n                t2 = ast.parse(\"\\n\".join(cand))\n                if _norm_dump(t2) == _norm_dump(tree):\n                    lines, tree, node = cand, t2, t2.body[0]\n            except SyntaxError:\n                pass\n    return lines, node\n\n\ndef reindent(lines: list[str], col: int) -> list[str]:\n    return [(\" \" * col + ln) if ln.strip() else \"\" for ln in lines]\n\n\n# ---------------------------------------------------------------- edit ops\ndef unified(rel, old, new, limit=MAX_OUT - 1500):\n    d = \"\".join(difflib.unified_diff(old.splitlines(True), new.splitlines(True), f\"a/{rel}\", f\"b/{rel}\", n=2))\n    return d if len(d) <= limit else d[:limit] + \"\\n... [diff truncated]\"\n\n\ndef check_compiles(src: str, rel: str):\n    try:\n        compile(src, rel, \"exec\", flags=ast.PyCF_ONLY_AST, dont_inherit=True)\n        compile(src, rel, \"exec\", dont_inherit=True)\n    except SyntaxError as e:\n        raise SymError(\"SyntaxError\", f\"edited file would not compile: {e.msg} (file line {e.lineno}); file left unchanged\",\n                       line=e.lineno, text=(e.text or \"\").rstrip()[:200])\n\n\ndef write_checked(root, rel, old_src, new_src):\n    if new_src == old_src:\n        raise SymError(\"NoChange\", \"new_source is identical to the current source; nothing written\")\n    check_compiles(new_src, rel)\n    with open(os.path.join(root, rel), \"w\", encoding=\"utf-8\") as f:\n        f.write(new_src)\n\n\ndef edit_symbol(root, symbol, new_source=None, allow_rename=False, old_string=None, new_string=None):\n    target = resolve(root, symbol)\n    old_src = read(root, target.rel)\n    if old_string is not None:\n        return _scoped_replace(root, target, old_src, old_string, new_string if new_string is not None else \"\")\n    if new_source is None:\n        raise SymError(\"MissingArgument\", \"give new_source (whole new definition) or old_string+new_string (change inside the symbol)\")\n    lines, node = parse_snippet(new_source)\n    tk, nk = target.kind, KIND[type(node)]\n    if (tk == \"class\") != (nk == \"class\"):\n        raise SymError(\"KindMismatch\", f\"{target.qname} is a {tk}; new_source defines a {nk}\")\n    if node.name != target.node.name and not str(allow_rename).lower() in (\"1\", \"true\", \"yes\"):\n        raise SymError(\"NameMismatch\", f\"new_source defines {node.name!r} but the target is {target.node.name!r}; \"\n                       \"pass allow_rename=true to rename, or use insert_symbol to add a new definition\")\n    file_lines = old_src.split(\"\\n\")\n    # keep existing decorators when the snippet has none\n    start = target.start if node.decorator_list else target.node.lineno\n    new_block = reindent(lines, target.col)\n    new_lines = file_lines[:start - 1] + new_block + file_lines[target.end:]\n    new_src = \"\\n\".join(new_lines)\n    write_checked(root, target.rel, old_src, new_src)\n    return {\"status\": \"ok\", \"symbol\": target.qname, \"file\": target.rel,\n            \"old_lines\": [start, target.end], \"new_lines\": [start, start + len(new_block) - 1],\n            \"diff\": unified(target.rel, old_src, new_src)}\n\n\ndef _scoped_replace(root, target, old_src, old_string, new_string):\n    \"\"\"Replace old_string inside the symbol's span only; uniqueness is checked within the symbol.\"\"\"\n    lines = old_src.split(\"\\n\")\n    a, b = target.start - 1, target.end\n    span = \"\\n\".join(lines[a:b])\n    n = span.count(old_string)\n    how = \"exact\"\n    if n == 0:   # tolerate indentation drift: match line-by-line on stripped text, re-indent replacement\n        ol = [x.strip() for x in old_string.strip(\"\\n\").split(\"\\n\")]\n        sl_ = lines[a:b]\n        hits = [i for i in range(len(sl_) - len(ol) + 1) if [x.strip() for x in sl_[i:i + len(ol)]] == ol]\n        if len(hits) == 1:\n            i = hits[0]\n            base = _indent(sl_[i])\n            nl_ = textwrap.dedent(new_string.strip(\"\\n\")).split(\"\\n\") if new_string.strip() else []\n            rep = [(\" \" * base + x) if x.strip() else \"\" for x in nl_]\n            new_span_lines = sl_[:i] + rep + sl_[i + len(ol):]\n            new_src = \"\\n\".join(lines[:a] + new_span_lines + lines[b:])\n            write_checked(root, target.rel, old_src, new_src)\n            return {\"status\": \"ok\", \"symbol\": target.qname, \"file\": target.rel, \"match\": \"indent_flexible\",\n                    \"diff\": unified(target.rel, old_src, new_src)}\n        n = len(hits)\n        how = \"flexible\"\n    if n == 0:\n        raise SymError(\"NoMatch\", f\"old_string not found inside {target.qname} (lines {target.start}-{target.end}); \"\n                       \"use show_symbol to see its current source\")\n    if n > 1:\n        raise SymError(\"MultipleMatches\", f\"old_string occurs {n} times inside {target.qname} ({how} match); add context lines\")\n    new_span = span.replace(old_string, new_string, 1)\n    new_src = \"\\n\".join(lines[:a] + new_span.split(\"\\n\") + lines[b:])\n    write_checked(root, target.rel, old_src, new_src)\n    return {\"status\": \"ok\", \"symbol\": target.qname, \"file\": target.rel, \"match\": \"exact\",\n            \"diff\": unified(target.rel, old_src, new_src)}\n\n\ndef insert_symbol(root, new_source, after=None, before=None, into=None):\n    lines, node = parse_snippet(new_source)\n    if sum(bool(x) for x in (after, before, into)) != 1:\n        raise SymError(\"BadArguments\", \"give exactly one of after=<symbol>, before=<symbol> or into=<class symbol | file.py>\")\n    blank_after = 0\n    if after or before:\n        anchor = resolve(root, after or before)\n        rel, col = anchor.rel, anchor.col\n        siblings = _siblings(root, anchor)\n        blank = 1 if anchor.parent is not None and anchor.parent.kind == \"class\" else 2\n        if after:\n            at = anchor.end\n        else:\n            at, blank_after, blank = anchor.start - 1, blank, 0\n    else:\n        path, dotted = _split_symbol(into)\n        if path and not dotted:\n            rel = path\n            if not os.path.isfile(os.path.join(root, rel)):\n                raise SymError(\"FileNotFound\", f\"no such file: {rel}\")\n            src = read(root, rel)\n            col, at, blank = 0, len(src.rstrip(\"\\n\").split(\"\\n\")), 2\n            siblings = [s.node.name for s in index_file(root, rel) if s.parent is None]\n        else:\n            cls = resolve(root, into, kinds={\"class\"})\n            rel, at, blank = cls.rel, cls.end, 1\n            col = cls.node.body[0].col_offset if cls.node.body else cls.col + 4\n            siblings = [n.name for n in cls.node.body if isinstance(n, DEF_TYPES)]\n    if node.name in siblings:\n        raise SymError(\"AlreadyExists\", f\"{node.name!r} already exists in that scope; use edit_symbol to change it\")\n    old_src = read(root, rel)\n    file_lines = old_src.split(\"\\n\")\n    body = reindent(lines, col)\n    new_block = [\"\"] * blank + body + [\"\"] * blank_after\n    new_lines = file_lines[:at] + new_block + file_lines[at:]\n    new_src = \"\\n\".join(new_lines)\n    write_checked(root, rel, old_src, new_src)\n    first = at + blank + 1\n    return {\"status\": \"ok\", \"inserted\": node.name, \"file\": rel, \"new_lines\": [first, first + len(body) - 1],\n            \"diff\": unified(rel, old_src, new_src)}\n\n\ndef _siblings(root, anchor):\n    if anchor.parent is None:\n        return [s.node.name for s in index_file(root, anchor.rel) if s.parent is None]\n    return [n.name for n in anchor.parent.node.body if isinstance(n, DEF_TYPES)]\n\n\ndef show_symbol(root, symbol, max_lines=150):\n    max_lines = int(max_lines)\n    path, dotted = _split_symbol(symbol)\n    if path and not dotted:                          # outline of a file\n        syms = index_file(root, path)\n        return {\"status\": \"ok\", \"file\": path, \"module\": module_name(path),\n                \"symbols\": [f\"{s.kind} {s.qname}  [{s.start}-{s.end}]\" for s in syms][:300]}\n    s = resolve(root, symbol)\n    src_lines = read(root, s.rel).split(\"\\n\")\n    body = src_lines[s.start - 1:s.end]\n    out = {\"status\": \"ok\", \"symbol\": s.qname, \"kind\": s.kind, \"file\": s.rel, \"lines\": [s.start, s.end]}\n    if s.gname != s.qname:\n        out[\"graph_name\"] = s.gname\n    if s.kind == \"class\":\n        out[\"members\"] = [f\"{KIND[type(n)]} {n.name}  [{n.lineno}-{n.end_lineno}]\" for n in s.node.body if isinstance(n, DEF_TYPES)]\n    if len(body) > max_lines:\n        body = body[:max_lines] + [f\"... [{len(src_lines[s.start - 1:s.end]) - max_lines} more lines; \"\n                                   \"show a member symbol instead]\"]\n        out[\"truncated\"] = True\n    out[\"source\"] = textwrap.dedent(\"\\n\".join(body))\n    return out\n\n\n# ---------------------------------------------------------------- CLI glue\ndef parse_args(argv):\n    \"\"\"Accept --key value pairs (ADK dict args) and bare positionals.\"\"\"\n    opts, pos, i = {}, [], 0\n    while i < len(argv):\n        a = argv[i]\n        if a == \"--\":\n            pos += argv[i + 1:]\n            break\n        if a.startswith(\"--\") and \"=\" in a:\n            k, v = a[2:].split(\"=\", 1)\n            opts[k.replace(\"-\", \"_\")] = v\n        elif a.startswith(\"--\") and i + 1 < len(argv):\n            opts[a[2:].replace(\"-\", \"_\")] = argv[i + 1]\n            i += 1\n        else:\n            pos.append(a)\n        i += 1\n    return opts, pos\n\n\ndef emit(obj):\n    s = json.dumps(obj, ensure_ascii=False)\n    if len(s) > MAX_OUT and \"source\" in obj:\n        obj[\"source\"] = obj[\"source\"][: max(500, MAX_OUT - (len(s) - len(obj[\"source\"])) - 200)] + \"\\n... [truncated]\"\n        obj[\"truncated\"] = True\n        s = json.dumps(obj, ensure_ascii=False)\n    print(s)\n\n\ndef run(fn, required, aliases=None):\n    opts, pos = parse_args(sys.argv[1:])\n    for k, alts in (aliases or {}).items():\n        for a in alts:\n            if a in opts and k not in opts:\n                opts[k] = opts.pop(a)\n    for k in required:\n        if k not in opts and pos:\n            opts[k] = pos.pop(0)\n    try:\n        missing = [k for k in required if k not in opts]\n        if missing:\n            raise SymError(\"MissingArgument\", f\"missing argument(s): {', '.join(missing)}\")\n        emit(fn(find_root(), **opts))\n    except SymError as e:\n        emit({\"status\": \"error\", \"error_type\": e.error_type, \"error_message\": e.message, **e.details})\n    except TypeError as e:\n        emit({\"status\": \"error\", \"error_type\": \"BadArguments\", \"error_message\": str(e)})\n",
"skills/graph-edit/scripts/show_symbol.py": "\"\"\"show_symbol: print one symbol's source (or a file's symbol outline) as JSON.\n\nargs: symbol=<pkg.mod.Class.method | path/file.py:Class.method | Class.method | path/file.py>\n      max_lines=<int, default 150>\n\"\"\"\nimport os\nimport sys\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport _symlib  # noqa: E402\n\n_symlib.run(_symlib.show_symbol, [\"symbol\"], aliases={\"symbol\": [\"name\", \"node\", \"qualified_name\", \"path\"]})\n",
"skills/graph-edit/scripts/edit_symbol.py": "\"\"\"edit_symbol: replace a whole def/class by name; validated with ast before writing.\n\nargs: symbol=<name as in show_symbol> and EITHER\n      new_source=<complete new definition>   (allow_rename=<true|false, default false>)\n      OR old_string=<text inside the symbol> new_string=<replacement>  (match scoped to the symbol)\n\"\"\"\nimport os\nimport sys\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport _symlib  # noqa: E402\n\n_symlib.run(_symlib.edit_symbol, [\"symbol\"],\n            aliases={\"symbol\": [\"name\", \"node\", \"qualified_name\"], \"new_source\": [\"source\", \"code\", \"new_code\"],\n                     \"old_string\": [\"old\", \"old_str\"], \"new_string\": [\"new\", \"new_str\"]})\n",
"skills/graph-edit/scripts/insert_symbol.py": "\"\"\"insert_symbol: add a NEW def/class next to an existing symbol, or at the end of a class/file.\n\nargs: new_source=<complete definition>  and exactly one of\n      after=<symbol> | before=<symbol> | into=<class symbol | path/file.py>\n\"\"\"\nimport os\nimport sys\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport _symlib  # noqa: E402\n\n_symlib.run(_symlib.insert_symbol, [\"new_source\"],\n            aliases={\"new_source\": [\"source\", \"code\", \"new_code\"]})\n",
"common/__init__.py": "",
"common/traces.py": "\"\"\"Parse swegemma ATIF v1.7 trace files (results/<run>/traces/trace_<id>.json) into flat tool events.\n\nATIF observations carry no tool_call_id, so each response is paired with the earliest\nunanswered call of the same tool name (FIFO), which matches the harness's emission order.\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field\nfrom typing import Any\n\n\n@dataclass\nclass ToolEvent:\n    idx: int\n    name: str\n    args: dict[str, Any]\n    result: Any = None          # parsed JSON dict when possible, else raw string/list\n    author: str | None = None\n\n\n@dataclass\nclass ParsedTrace:\n    events: list[ToolEvent] = field(default_factory=list)\n    user_messages: list[str] = field(default_factory=list)\n    final_metrics: dict[str, Any] | None = None\n\n\ndef _parse_json(x: Any) -> Any:\n    if isinstance(x, str):\n        s = x.strip()\n        if s.startswith(\"{\"):\n            try:\n                return json.loads(s)\n            except json.JSONDecodeError:\n                return x\n    if isinstance(x, list) and len(x) == 1 and isinstance(x[0], dict) and \"text\" in x[0]:\n        return _parse_json(x[0][\"text\"])\n    return x\n\n\ndef _iter_steps(traj: dict):\n    for st in traj.get(\"steps\", []):\n        yield st\n    for sub in traj.get(\"subagent_trajectories\", []) or []:\n        yield from _iter_steps(sub)\n\n\ndef parse_trace(path_or_dict: str | dict) -> ParsedTrace:\n    traj = path_or_dict\n    if isinstance(path_or_dict, str):\n        with open(path_or_dict, encoding=\"utf-8\") as f:\n            traj = json.load(f)\n    out = ParsedTrace(final_metrics=traj.get(\"final_metrics\"))\n    pending: dict[str, list[ToolEvent]] = {}\n    for st in _iter_steps(traj):\n        if st.get(\"source\") == \"user\" and isinstance(st.get(\"message\"), str):\n            out.user_messages.append(st[\"message\"])\n        for tc in st.get(\"tool_calls\", []) or []:\n            ev = ToolEvent(len(out.events), tc.get(\"function_name\", \"\"), tc.get(\"arguments\") or {},\n                           author=(tc.get(\"extra\") or {}).get(\"author\"))\n            out.events.append(ev)\n            pending.setdefault(ev.name, []).append(ev)\n        obs = st.get(\"observation\")\n        if obs is not None:\n            name = (obs.get(\"extra\") or {}).get(\"tool_name\")\n            q = pending.get(name) or []\n            if q:\n                q.pop(0).result = _parse_json(obs.get(\"content\"))\n    return out\n",
"common/metrics.py": "\"\"\"Shared per-task metrics, identical across pilots.\n\nresolved, wall_s, tool_calls, dup_call_share (same tool + same args as an earlier call),\nmalformed_share, failed_edits, edit_attempts, skill_invocations.\n\nDefinitions:\n  malformed call: the harness/ADK rejected the call itself, not its effect: unknown tool,\n    argument validation/type errors, or a skill-tool argument error. Truncated tool calls never\n    reach the trace as calls, so they are counted separately from \"cut off\" nudges.\n  edit attempt: edit_file, write_file, or a skill script whose name contains \"edit\"/\"insert\".\n  failed edit: an edit attempt whose result has status \"error\" (or, for skills, an error JSON\n    in stdout).\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nfrom typing import Any\n\nfrom .traces import ParsedTrace, ToolEvent\n\nMALFORMED_ERROR_TYPES = {\"ValueError\", \"TypeError\", \"ValidationError\", \"KeyError\", \"ToolNotFoundError\",\n                         \"InvalidArgument\", \"JSONDecodeError\", \"pydantic_core._pydantic_core.ValidationError\"}\nEDIT_TOOLS = {\"edit_file\", \"write_file\"}\nSKILL_TOOLS = {\"load_skill\", \"load_skill_resource\", \"run_skill_script\", \"list_skills\"}\nCUTOFF_MARK = \"was cut off\"\n# error types a skill script may print for a call whose arguments were unusable (convention: JSON on stdout)\nSKILL_ARG_ERRORS = {\"MissingArgument\", \"BadArguments\"}\n\n\ndef _canon(args: dict) -> str:\n    return json.dumps(args, sort_keys=True, ensure_ascii=False, default=str)\n\n\ndef skill_stdout_json(ev: ToolEvent) -> dict | None:\n    r = ev.result\n    if isinstance(r, dict) and isinstance(r.get(\"stdout\"), str):\n        s = r[\"stdout\"].strip()\n        if s.startswith(\"{\"):\n            try:\n                return json.loads(s.splitlines()[0] if s.count(\"\\n\") and not s.endswith(\"}\") else s)\n            except json.JSONDecodeError:\n                return None\n    return None\n\n\ndef is_error(ev: ToolEvent) -> bool:\n    r = ev.result\n    if isinstance(r, dict):\n        if r.get(\"status\") == \"error\" or \"error_code\" in r:\n            return True\n        j = skill_stdout_json(ev)\n        return bool(j and j.get(\"status\") == \"error\")\n    return False\n\n\ndef is_malformed(ev: ToolEvent) -> bool:\n    r = ev.result\n    if not isinstance(r, dict):\n        return False\n    et = str(r.get(\"error_type\", \"\"))\n    if et in MALFORMED_ERROR_TYPES:\n        return True\n    if r.get(\"error_code\") in (\"INVALID_ARGUMENTS\", \"MISSING_SKILL_NAME\", \"SKILL_NOT_FOUND\", \"MISSING_FILE_PATH\"):\n        return True\n    j = skill_stdout_json(ev)\n    if j and j.get(\"error_type\") in SKILL_ARG_ERRORS:\n        return True\n    msg = str(r.get(\"error\", \"\")) + str(r.get(\"error_message\", \"\"))\n    return \"not found in tools_dict\" in msg or \"missing\" in msg.lower() and \"argument\" in msg.lower()\n\n\ndef is_edit(ev: ToolEvent) -> bool:\n    if ev.name in EDIT_TOOLS:\n        return True\n    if ev.name == \"run_skill_script\":\n        fp = str(ev.args.get(\"file_path\", \"\"))\n        a = ev.args.get(\"args\")\n        sub = (a[0] if isinstance(a, list) and a else \"\") if not isinstance(a, dict) else a.get(\"op\", \"\")\n        return any(k in (fp + \" \" + str(sub)) for k in (\"edit\", \"insert\"))\n    return False\n\n\ndef task_metrics(trace: ParsedTrace, result_row: dict | None = None) -> dict[str, Any]:\n    evs = trace.events\n    seen, dups = set(), 0\n    for ev in evs:\n        k = (ev.name, _canon(ev.args))\n        dups += k in seen\n        seen.add(k)\n    edits = [e for e in evs if is_edit(e)]\n    n = len(evs)\n    row = result_row or {}\n    return {\n        \"instance_id\": row.get(\"instance_id\"),\n        \"resolved\": row.get(\"resolved\"),\n        \"wall_s\": row.get(\"duration_seconds\"),\n        \"tool_calls\": n,\n        \"harness_tool_calls\": row.get(\"tool_calls\"),\n        \"dup_call_share\": dups / n if n else 0.0,\n        \"malformed_calls\": sum(is_malformed(e) for e in evs),\n        \"malformed_share\": sum(is_malformed(e) for e in evs) / n if n else 0.0,\n        \"cutoff_nudges\": sum(CUTOFF_MARK in m for m in trace.user_messages),\n        \"edit_attempts\": len(edits),\n        \"failed_edits\": sum(is_error(e) for e in edits),\n        \"failed_edit_share\": (sum(is_error(e) for e in edits) / len(edits)) if edits else None,\n        \"skill_invocations\": sum(e.name in SKILL_TOOLS for e in evs),\n        \"skill_script_runs\": sum(e.name == \"run_skill_script\" for e in evs),\n        \"submitted\": any(e.name == \"submit_patch\" for e in evs),\n        \"patch_size\": row.get(\"agent_patch_size\"),\n    }\n\n\ndef aggregate(rows: list[dict]) -> dict[str, Any]:\n    n = len(rows)\n    if not n:\n        return {}\n    s = lambda k: sum((r.get(k) or 0) for r in rows)  # noqa: E731\n    edits = s(\"edit_attempts\")\n    calls = s(\"tool_calls\")\n    return {\n        \"tasks\": n,\n        \"resolved\": s(\"resolved\"),\n        \"resolved_rate\": s(\"resolved\") / n,\n        \"mean_wall_s\": s(\"wall_s\") / n,\n        \"mean_tool_calls\": calls / n,\n        \"dup_call_share\": sum(r[\"dup_call_share\"] * r[\"tool_calls\"] for r in rows) / calls if calls else 0.0,\n        \"malformed_share\": s(\"malformed_calls\") / calls if calls else 0.0,\n        \"cutoff_nudges\": s(\"cutoff_nudges\"),\n        \"edit_attempts\": edits,\n        \"failed_edits\": s(\"failed_edits\"),\n        \"failed_edit_share\": s(\"failed_edits\") / edits if edits else None,\n        \"skill_invocations\": s(\"skill_invocations\"),\n        \"tasks_using_skill\": sum(1 for r in rows if r.get(\"skill_script_runs\")),\n        \"empty_patches\": sum(1 for r in rows if not r.get(\"patch_size\")),\n    }\n"
}
for rel, txt in EMBEDDED.items():
    p = PILOT / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(txt, encoding='utf-8')
sys.path.insert(0, str(PILOT))
PROMPT_ADDON = '\n\n## Editing code by symbol name (graph-edit skill)\nTo change code inside a function, method or class, prefer the `graph-edit` skill over edit_file:\n- See a symbol: run_skill_script(skill_name="graph-edit", file_path="scripts/show_symbol.py", args={"symbol": "pkg.module.Class.method"})\n- Change part of it: run_skill_script(skill_name="graph-edit", file_path="scripts/edit_symbol.py", args={"symbol": "pkg.module.Class.method", "old_string": "...", "new_string": "..."})\n- Replace it whole: same script with args={"symbol": ..., "new_source": "<complete new definition>"}\n- Add a new function/method: scripts/insert_symbol.py with args={"new_source": ..., "after": "<existing symbol>"}\nSymbol names are the dotted names that get_code_neighbors / search_similar_code print. Edits are syntax-checked; on "status": "error" the file is unchanged.\nUse edit_file only for imports and other module-level lines.\n'
print(sorted(EMBEDDED))

In [ ]:
# Build the two submissions from sample_submission (same model, budgets and sampling; only the skill differs)
from swegemma.models import load_tasks
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
def build_submission(cond):
    d = WORKING_DIR / f'sub_{cond}'
    if d.exists(): shutil.rmtree(d)
    shutil.copytree(DATA_DIR / 'sample_submission', d)
    shutil.rmtree(d / 'adapters', ignore_errors=True)            # 31b-shaped dummy LoRAs
    for y in [d / 'agent.yaml', *sorted((d / 'sub_agents').glob('*.yaml'))]:
        t = y.read_text()
        t = re.sub(r'^(\s*model:\s*).*$', rf'\g<1>{MODEL_NAME}', t, flags=re.M)
        t = re.sub(r'^\s*adapter:\s*.*\n?', '', t, flags=re.M)
        y.write_text(t)
    samp = d / 'configs' / 'sampling.yaml'
    samp.write_text(samp.read_text().rstrip() + f'\nseed: {SEED}\n')
    (d / 'eval_config.yaml').write_text('evaluation:\n' + ''.join(f'  {k}: {v}\n' for k, v in BUDGET.items()))
    if cond == 'graph_edit':
        shutil.copytree(PILOT / 'skills', d / 'skills')
        y = d / 'agent.yaml'
        y.write_text(y.read_text().rstrip() + '\nskills:\n  - skills/graph-edit\n')
        p = d / 'prompts' / 'system.md'
        p.write_text(p.read_text() + PROMPT_ADDON)
    for pc in d.rglob('__pycache__'): shutil.rmtree(pc)
    return d
SUBS = {c: build_submission(c) for c in CONDITIONS}
tasks = load_tasks(DATA_DIR / 'tasks.jsonl')
sel = [t for t in tasks if t.instance_id in set(TASK_IDS)]
assert len(sel) == len(TASK_IDS), (len(sel), TASK_IDS)
print({c: str(p) for c, p in SUBS.items()}, len(sel), 'tasks')
print((SUBS['graph_edit'] / 'agent.yaml').read_text())

In [ ]:
# Validate both submissions with the official validator before spending GPU time on vLLM
from adk_submission import validate_directory
from swegemma.config import build_submission_limits
limits, gen_constraints = build_submission_limits()
for c, d in SUBS.items():
    validate_directory(d, limits=limits); print(c, 'valid')

In [ ]:
# vLLM server (as in the official starter)
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
litellm.drop_params = True
MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{MODEL_NAME}/2')
assert MODEL_PATH.exists(), MODEL_PATH
adapters = discover_adapters(str(SUBS['baseline']), adapter_extensions={'.safetensors'})
gpu_count = torch.cuda.device_count()
tp = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
T_VLLM = time.time()
server_instance = VllmServer(VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto', gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True, enable_lora=False, max_loras=1, max_lora_rank=128, scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp, startup_timeout=60 * 20), adapter_manifest=adapters)
server_instance.start()
models = server_instance.create_model_registry(aliases=[MODEL_NAME], model_prefix='openai/', api_key=os.environ.get('LOCAL_API_KEY', 'EMPTY'))
T_VLLM = time.time() - T_VLLM
print('vLLM up in', round(T_VLLM), 's, tp =', tp)

In [ ]:
# Run both conditions on the same tasks
import asyncio, concurrent.futures
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig
from swegemma.evaluate import Evaluator
def run_sync(fn):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())
WALL = {}
for cond in CONDITIONS:
    if (time.time() - T0) / 3600 > HARD_STOP_H:
        print('HARD STOP: not starting', cond); break
    res_dir = WORKING_DIR / f'results_{cond}'
    if res_dir.exists(): shutil.rmtree(res_dir)
    cfg = EvalConfig(
        tasks_path=DATA_DIR / 'tasks.jsonl', snapshots_dir=DATA_DIR / 'snapshots', results_dir=res_dir,
        submission_dir=SUBS[cond], models=models, sandbox='subprocess',
        timeout_seconds=BUDGET['timeout_seconds'], verification_timeout_seconds=300,
        max_time_minutes=BUDGET['max_time_minutes'], max_tool_calls=BUDGET['max_tool_calls'], max_turns=BUDGET['max_turns'],
        limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'), wheels_dir=DATA_DIR / 'wheels',
        task_ids=list(TASK_IDS), concurrency=CONCURRENCY, display_mode='quiet', verbose=False)
    t = time.time()
    run_sync(Evaluator(cfg).run)
    WALL[cond] = time.time() - t
    print(cond, 'done in', round(WALL[cond] / 60, 1), 'min')

In [ ]:
# Metrics (common/metrics.py, identical across pilots) -> one JSON with aggregates + per-task numbers
from common.traces import parse_trace
from common.metrics import task_metrics, aggregate
out = {'smoke': SMOKE, 'model': MODEL_NAME, 'task_ids': list(TASK_IDS), 'budget': BUDGET, 'concurrency': CONCURRENCY,
       'seed': SEED, 'vllm_startup_s': T_VLLM, 'condition_wall_s': WALL, 'conditions': {}}
for cond in [c for c in CONDITIONS if c in WALL]:
    res_dir = WORKING_DIR / f'results_{cond}'
    rows = [json.loads(l) for l in open(res_dir / 'task_results.jsonl')]
    per = []
    for r in rows:
        tp_ = res_dir / 'traces' / f"trace_{r['instance_id']}.json"
        tr = parse_trace(str(tp_)) if tp_.exists() else parse_trace({'steps': []})
        m = task_metrics(tr, r)
        m['error'] = (r.get('error') or '')[:120]
        m['tool_names'] = dict(__import__('collections').Counter(e.name for e in tr.events))
        m['skill_scripts'] = dict(__import__('collections').Counter(str(e.args.get('file_path')) for e in tr.events if e.name == 'run_skill_script'))
        m['skill_errors'] = dict(__import__('collections').Counter(
            ((__import__('common.metrics', fromlist=['x']).skill_stdout_json(e) or {}).get('error_type') or 'ok')
            for e in tr.events if e.name == 'run_skill_script'))
        m['edit_file_errors'] = dict(__import__('collections').Counter(
            (e.result.get('error_type') if isinstance(e.result, dict) else 'n/a') for e in tr.events
            if e.name == 'edit_file' and isinstance(e.result, dict) and e.result.get('status') == 'error'))
        per.append(m)
    out['conditions'][cond] = {'aggregate': aggregate(per), 'tasks': per}
elapsed_h = (time.time() - T0) / 3600
out['notebook_wall_h'] = elapsed_h
if SMOKE:
    per_task_min = {c: WALL[c] / 60 / len(TASK_IDS) for c in WALL}
    full_h = (T_VLLM + sum(WALL[c] * 15 / len(TASK_IDS) for c in WALL)) / 3600 + 0.25
    out['extrapolation'] = {'minutes_per_task_at_concurrency': per_task_min, 'full_run_L4x4_hours_est': full_h}
    print('SMOKE: per-task minutes (at concurrency', CONCURRENCY, '):', per_task_min)
    print('Estimated full run (15 tasks x 2 conditions incl. setup): %.2f L4x4 hours' % full_h)
with open(WORKING_DIR / 'pilot_b_live_results.json', 'w') as f:
    json.dump(out, f, indent=1, default=str)
for c in out['conditions']: print(c, json.dumps(out['conditions'][c]['aggregate']))
print('notebook wall: %.2f h' % elapsed_h)